In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import MinMaxScaler

In [3]:
orders = pd.read_csv("olist_orders_dataset_clean.csv")
items = pd.read_csv("olist_order_items_dataset_clean.csv")
payments = pd.read_csv("olist_order_payments_dataset_clean.csv")
reviews = pd.read_csv("olist_order_reviews_dataset_clean.csv")
customers = pd.read_csv("olist_customers_dataset_clean.csv")

In [4]:
df = orders.merge(reviews, on="order_id", how="left")
df = df.merge(customers, on="customer_id", how="left")

In [6]:
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

In [7]:
item_count = items.groupby('order_id').size().reset_index(name='raw_item_count')
df = df.merge(item_count, on='order_id', how='left')

In [8]:
order_value = items.groupby('order_id')['price'].sum().reset_index()
order_value.rename(columns={'price':'raw_order_value'}, inplace=True)

df = df.merge(order_value, on='order_id', how='left')

In [9]:
freight_total = items.groupby('order_id')['freight_value'].sum().reset_index()
freight_total.rename(columns={'freight_value':'raw_freight_total'}, inplace=True)

df = df.merge(freight_total, on='order_id', how='left')

In [10]:
df['raw_freight_ratio'] = df['raw_freight_total'] / df['raw_order_value']

In [11]:
df['raw_delivery_time'] = (
    df['order_delivered_customer_date'] -
    df['order_purchase_timestamp']
).dt.days

In [12]:
df['raw_estimated_delivery'] = (
    df['order_estimated_delivery_date'] -
    df['order_purchase_timestamp']
).dt.days

In [13]:
df['raw_delivery_delay'] = (
    df['order_delivered_customer_date'] -
    df['order_estimated_delivery_date']
).dt.days

In [14]:
df['raw_seller_processing_time'] = (
    df['order_delivered_carrier_date'] -
    df['order_purchase_timestamp']
).dt.days

In [15]:
customer_orders = orders.groupby('customer_id').size().reset_index()

customer_orders.rename(columns={0:'raw_customer_orders'}, inplace=True)

df = df.merge(customer_orders, on='customer_id', how='left')

In [17]:
raw_cols = [c for c in df.columns if c.startswith('raw_')]

df[raw_cols] = df[raw_cols].fillna(df[raw_cols].median())

In [18]:
for col in raw_cols:
    
    q1 = df[col].quantile(0.01)
    q3 = df[col].quantile(0.99)

    df[col] = np.clip(df[col], q1, q3)

In [19]:
scaler = MinMaxScaler()

norm_cols = [c.replace('raw_', 'norm_') for c in raw_cols]

df[norm_cols] = scaler.fit_transform(df[raw_cols])

In [20]:
df[norm_cols]

,norm_item_count,norm_order_value,norm_freight_total,norm_freight_ratio,norm_delivery_time,norm_estimated_delivery,norm_delivery_delay,norm_seller_processing_time,norm_customer_orders
0,0.0,0.018259,0.013663,0.185964,0.159091,0.204545,0.518519,0.117647,0.0
1,0.0,0.108244,0.157900,0.116652,0.272727,0.295455,0.555556,0.058824,0.0
2,0.0,0.150036,0.121533,0.066573,0.181818,0.454545,0.333333,0.000000,0.0
3,0.0,0.033485,0.203513,0.405535,0.272727,0.454545,0.425926,0.176471,0.0
4,0.0,0.008024,0.013663,0.289161,0.022727,0.136364,0.481481,0.000000,0.0
...,...,...,...,...,...,...,...,...,...
99987,0.0,0.060873,0.058455,0.109598,0.159091,0.272727,0.462963,0.058824,0.0
99988,0.0,0.165252,0.130573,0.062879,0.477273,0.386364,0.629630,0.058824,0.0
99989,0.0,0.196789,0.592048,0.203382,0.522727,0.545455,0.555556,0.058824,0.0
99990,0.5,0.352993,0.758065,0.140290,0.363636,0.704545,0.277778,0.176471,0.0


In [22]:
df['target'] = df['review_score'].apply(
    lambda x: 0 if x <= 2 else (1 if x == 3 else 2)
)

In [23]:
features = norm_cols

X = df[features]

y = df['target']

In [24]:
print(y.value_counts(normalize=True))

target
2    0.772442
0    0.145762
1    0.081797
Name: proportion, dtype: float64


In [25]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

X_res, y_res = smote.fit_resample(X, y)

In [26]:
from collections import Counter
print(Counter(y_res))

Counter({2: 77238, 0: 77238, 1: 77238})


In [27]:
import xgboost as xgb

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

In [28]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [29]:
model = xgb.XGBClassifier(
    objective='multi:softprob',
    num_class=3,
    eval_metric='mlogloss',
    max_depth=6,
    learning_rate=0.1,
    n_estimators=200,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

In [30]:
model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=200, n_jobs=None, num_class=3, ...)

In [31]:
y_pred = model.predict(X_test)

In [32]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.70      0.40      0.51      2915
           1       0.00      0.00      0.00      1636
           2       0.82      0.98      0.89     15448

    accuracy                           0.81     19999
   macro avg       0.51      0.46      0.47     19999
weighted avg       0.74      0.81      0.76     19999



In [33]:
df['target'] = df['review_score'].apply(lambda x: 1 if x >= 4 else 0)

In [34]:
features = norm_cols

X = df[features]
y = df['target']

In [35]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [36]:
import xgboost as xgb

model = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    max_depth=6,
    learning_rate=0.1,
    n_estimators=200,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=200, n_jobs=None,
              num_parallel_tree=None, ...)

In [37]:
y_pred = model.predict(X_test)


In [38]:
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.78      0.32      0.46      4705
           1       0.82      0.97      0.89     15294

    accuracy                           0.82     19999
   macro avg       0.80      0.65      0.68     19999
weighted avg       0.81      0.82      0.79     19999

